# Positive–Unlabeled Learning for ADC Target Discovery

**v0.1 educational notebook**

## Goal
Reframe target discovery as Positive–Unlabeled (PU) learning and estimate both candidate scores and sampling uncertainty.

## Canonical guide

### Learning objectives
- Model known-positive versus unlabeled data more realistically.
- Estimate candidate rank and sampling uncertainty using bagging.
- Use disagreement to separate exploitation and exploration candidates.

### Data mode
Canonical target features, educational positive labels, and earlier rule/ML scores.

### Inputs
`outputs/adc_target_feature_matrix_v01.csv`, `outputs/adc_ml_ranking_v01.csv`, `outputs/adc_rule_based_ranking_v01.csv`, `data/example/known_adc_targets_v01.csv`

### Canonical outputs
`outputs/adc_pu_ranking_v01.csv`, `outputs/adc_rule_supervised_pu_comparison_v01.csv`, `outputs/adc_pu_feature_importance_v01.csv`, `outputs/adc_experimental_batch_001.csv`

### Scientific limitation
The PU score is not a calibrated probability of ADC success. The positive set is intentionally small, and bagging variation is only an uncertainty proxy.

### Next step
Notebook 08 treats the selected batch as if functional wet-lab results had returned.


In [1]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'
OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)
print('ROOT:', ROOT.resolve())

ROOT: /mnt/data/audit_adc/adc-scientific-ai


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
f=pd.read_csv(OUTPUTS/'adc_target_feature_matrix_v01.csv',index_col=0)
labels=pd.read_csv(DATA/'example'/'known_adc_targets_v01.csv'); known=set(labels.gene_symbol)
cols=['tumor_score','selectivity_score','prevalence_score','surface_score_norm','heterogeneity_risk','normal_rna_risk','normal_protein_risk_norm','critical_organ_risk']
d=f[f.surface_score.ge(.5)].dropna(subset=cols).copy(); d['is_positive']=d.index.isin(known)
print('positives',d.is_positive.sum(),'unlabeled',(~d.is_positive).sum())

positives 4 unlabeled 20


In [3]:
def bagging_pu(data,cols,n_iterations=25,negative_ratio=4,seed=42):
    rng=np.random.default_rng(seed); pos=data.index[data.is_positive]; unl=data.index[~data.is_positive]; npos=len(pos); nsamp=min(npos*negative_ratio,len(unl)); scores=[]; imps=[]
    for i in range(n_iterations):
        neg=rng.choice(unl,size=nsamp,replace=False); idx=list(pos)+list(neg); y=np.r_[np.ones(npos),np.zeros(nsamp)]
        m=RandomForestClassifier(n_estimators=80,min_samples_leaf=2,class_weight='balanced',random_state=seed+i,n_jobs=1)
        m.fit(data.loc[idx,cols],y); scores.append(m.predict_proba(data[cols])[:,1]); imps.append(m.feature_importances_)
    S=np.vstack(scores); I=np.vstack(imps)
    return S.mean(0),S.std(0),I.mean(0),I.std(0)
mean,std,imp_mean,imp_std=bagging_pu(d,cols)
d['bagging_pu_score']=mean; d['bagging_pu_uncertainty']=std; d['robust_pu_score']=mean-std
ranking=d.sort_values('robust_pu_score',ascending=False)
ranking[['is_positive','bagging_pu_score','bagging_pu_uncertainty','robust_pu_score']].head(12)

,is_positive,bagging_pu_score,bagging_pu_uncertainty,robust_pu_score
gene_symbol,,,,
NECTIN4,True,0.921881,0.021809,0.900071
FOLR1,True,0.884603,0.029510,0.855093
ERBB2,True,0.867246,0.040116,0.827130
TACSTD2,True,0.848475,0.042575,0.805899
EPCAM,False,0.261748,0.059973,0.201775
GENE_R,False,0.209528,0.047970,0.161558
GENE_A,False,0.184128,0.044254,0.139874
GENE_G,False,0.205692,0.074702,0.130990
GENE_H,False,0.076165,0.031361,0.044804


In [4]:
sup=pd.read_csv(OUTPUTS/'adc_ml_ranking_v01.csv',index_col=0); rule=pd.read_csv(OUTPUTS/'adc_rule_based_ranking_v01.csv',index_col=0)
comparison=pd.DataFrame(index=d.index)
comparison['supervised_score']=sup.reindex(d.index).ensemble_score
comparison['pu_score']=d.bagging_pu_score
comparison['pu_uncertainty']=d.bagging_pu_uncertainty
comparison['rule_score']=rule.reindex(d.index).adc_target_score
comparison['three_way_consensus']=comparison[['rule_score','supervised_score','pu_score']].mean(axis=1)
comparison['three_way_disagreement']=comparison[['rule_score','supervised_score','pu_score']].std(axis=1)
importance=pd.DataFrame({'feature':cols,'importance_mean':imp_mean,'importance_std':imp_std}).sort_values('importance_mean',ascending=False)
ranking.to_csv(OUTPUTS/'adc_pu_ranking_v01.csv')
comparison.to_csv(OUTPUTS/'adc_rule_supervised_pu_comparison_v01.csv')
importance.to_csv(OUTPUTS/'adc_pu_feature_importance_v01.csv',index=False)

In [5]:
# 8 exploitation + 4 exploration; avoid duplicate genes across groups
score_cut=ranking.bagging_pu_score.quantile(.75); unc_cut=ranking.bagging_pu_uncertainty.quantile(.60)
exploit=ranking[(ranking.bagging_pu_score>=score_cut)&(ranking.bagging_pu_uncertainty<unc_cut)].head(8).copy(); exploit['selection_strategy']='EXPLOITATION'
remaining=ranking.drop(index=exploit.index)
explore=remaining[(remaining.bagging_pu_score>=ranking.bagging_pu_score.quantile(.50))].sort_values('bagging_pu_uncertainty',ascending=False).head(4).copy(); explore['selection_strategy']='EXPLORATION'
batch=pd.concat([exploit,explore]); batch.to_csv(OUTPUTS/'adc_experimental_batch_001.csv')
print('batch size:',len(batch)); batch[['selection_strategy','bagging_pu_score','bagging_pu_uncertainty']]

batch size: 6


,selection_strategy,bagging_pu_score,bagging_pu_uncertainty
gene_symbol,,,
NECTIN4,EXPLOITATION,0.921881,0.021809
FOLR1,EXPLOITATION,0.884603,0.029510
GENE_G,EXPLORATION,0.205692,0.074702
EPCAM,EXPLORATION,0.261748,0.059973
GENE_R,EXPLORATION,0.209528,0.047970
GENE_A,EXPLORATION,0.184128,0.044254


## Interpretation
The PU score is a relative “known-positive-like” ranking signal, not a calibrated probability that a target will succeed. High-score/low-uncertainty candidates support exploitation; high-uncertainty candidates can support exploration.